# Assignment-4
## UCS420 - Cognitive Computing
### Cognitive FAQ System Using Pandas (Nova 2.0)

**Name:**   Kashish Singhal
**Roll No:** 1024170144


In [ ]:
import pandas as pd


## Q1: Build Personalized Knowledge Base

Roll number = 1024160131  
Last two digits = 3 and 1

- digit 3 -> 3 % 3 = 0 -> billing  
- digit 1 -> 1 % 3 = 1 -> account


In [ ]:
roll_no = "1024160131"
print("Roll number:", roll_no)
print("Digits:", list(roll_no))

categories = ["billing", "account", "general"]
d1 = int(roll_no[-2])  # 3
d2 = int(roll_no[-1])  # 1
print("Last two digits:", d1, d2)
print("Category for", d1, "->", categories[d1 % 3])
print("Category for", d2, "->", categories[d2 % 3])


In [ ]:
fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.",
     "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.",
     "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.",
     "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.",
     "keywords": "pay payment upi fee", "category": "billing"},
]

# personalized from roll digits (billing + account)
personal_entries = [
    {"question": "how do i get a fee refund",
     "answer": "Login to the student portal, go to Payments and submit a refund request with your receipt.",
     "keywords": "refund fee payment portal receipt",
     "category": "billing"},
    {"question": "how do i update my registered mobile number",
     "answer": "Open Profile > Contact Details, enter the new number and verify with OTP.",
     "keywords": "mobile number update profile otp",
     "category": "account"},
]

faq_df = pd.DataFrame(fixed_entries + personal_entries)
print("Final FAQ DataFrame:")
faq_df


## Q2: Generate and Score a Hypothesis


In [ ]:
def score_query(query, df):
    query_words = query.lower().split()
    results = []

    for idx, row in df.iterrows():
        kw = row["keywords"].lower().split()
        # count how many query words match keywords
        score = 0
        for w in query_words:
            if w in kw:
                score += 1
        if score > 0:
            results.append({
                "index": idx,
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "score": score
            })

    # rank by confidence (score)
    results = sorted(results, key=lambda x: x["score"], reverse=True)
    return results


# testing
q = "how to pay fee using upi"
matched = score_query(q, faq_df)
print("Query:", q)
print()
for m in matched:
    print("Score:", m["score"], "|", m["question"], "->", m["answer"])


## Q3: same_category function


In [ ]:
def same_category(category_name, df):
    return df[df["category"] == category_name]["question"]


# using category of one personalized entry (billing)
print("Questions in billing category:")
print(same_category("billing", faq_df))


## Q4: Add keyword to an entry and save CSV


In [ ]:
# picking the password reset entry (index 1)
print("Selected entry:")
print(faq_df.loc[1])
print()

new_kw = input("Enter a new keyword to add: ")
faq_df.at[1, "keywords"] = faq_df.at[1, "keywords"] + " " + new_kw.strip()

csv_name = "1024160131_faq_data.csv"
faq_df.to_csv(csv_name, index=False)
print("Updated keywords:", faq_df.at[1, "keywords"])
print("Saved to", csv_name)
faq_df


## Q5: FAQ count per category (groupby)


In [ ]:
print(faq_df.groupby("category").size())
print()
print(faq_df.groupby("category")["question"].count())


## Q6: Scoring with tie handling

If two or more entries have the same highest score, print all of them.


In [ ]:
def score_query_with_ties(query, df):
    query_words = query.lower().split()
    results = []

    for idx, row in df.iterrows():
        kw = row["keywords"].lower().split()
        score = 0
        for w in query_words:
            if w in kw:
                score += 1
        if score > 0:
            results.append({
                "index": idx,
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "score": score
            })

    if len(results) == 0:
        print("No match found.")
        return []

    results = sorted(results, key=lambda x: x["score"], reverse=True)
    top = results[0]["score"]
    ties = [r for r in results if r["score"] == top]

    if len(ties) > 1:
        print("Tie found! Multiple entries have score =", top)
        for t in ties:
            print("-", t["question"], "|", t["answer"], "| score:", t["score"])
    else:
        print("Best match (no tie):")
        t = ties[0]
        print("-", t["question"], "|", t["answer"], "| score:", t["score"])

    return ties


print("=== Query that should TIE (both billing fee entries) ===")
score_query_with_ties("fee", faq_df)

print()
print("=== Query that should NOT tie ===")
score_query_with_ties("password reset login", faq_df)
